# Understanding IFC Files: Building Code Compliance Exercises

## What is IFC?

**IFC (Industry Foundation Classes)** is a standard data format for building information. It stores geometric, spatial, and semantic information about buildings in a structured way. Think of it as a database format for architectural and engineering data.

Industries use IFC to:
- Share building models across different software
- Extract specific information (spaces, materials, systems)
- Validate designs against building codes
- Run simulations and analysis

## Resources

Before starting, explore these resources to understand IFC better:

- **IFC Knowledge Base**: https://notebooklm.google.com/notebook/0925c2a1-519b-40a8-aca4-1e832d219f3c
- **BuildingSmart (IFC Standard)**: https://www.buildingsmart.org/standards/bsi-standards/industry-foundation-classes/
- **IfcOpenShell Python Docs**: https://docs.ifcopenshell.org/ifcopenshell-python.html
- **Catalan Building Code Reference**: https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e (for exercises 1 & bonus)

This notebook uses **IfcOpenShell**, a Python library for reading and writing IFC files.

## Setup: Load and Explore IFC Files

First, let's install IfcOpenShell and load the duplex apartment model.

In [1]:
# Install IfcOpenShell (run this once)
import subprocess
import sys

try:
    import ifcopenshell
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ifcopenshell", "-q"])
    import ifcopenshell

# Import other useful libraries
import json
from pathlib import Path
from collections import defaultdict

# Load the IFC file
ifc_file_path = Path("assets/duplex.ifc")
ifc = ifcopenshell.open(ifc_file_path)

print(f"✓ IFC file loaded: {ifc_file_path}")
print(f"✓ IFC Schema: {ifc.schema}")

✓ IFC file loaded: assets\duplex.ifc
✓ IFC Schema: IFC2X3


### What's Inside the File?

Let's explore the structure of the IFC model:

In [2]:
# Get all unique entity types in the model
all_entities = list(ifc)
entity_types = defaultdict(int)
for entity in all_entities:
    entity_types[entity.is_a()] += 1

# Show entity counts
print("Entity types in this model:")
print("-" * 40)
for entity_type in sorted(entity_types.keys()):
    count = entity_types[entity_type]
    print(f"  {entity_type}: {count}")

print(f"\nTotal entities: {len(all_entities)}")

# Find basic info
building = ifc.by_type("IfcBuilding")[0] if ifc.by_type("IfcBuilding") else None
if building:
    print(f"\nBuilding name: {building.Name}")
    print(f"Building description: {building.Description}")

Entity types in this model:
----------------------------------------
  IfcApplication: 1
  IfcArbitraryClosedProfileDef: 102
  IfcArbitraryOpenProfileDef: 184
  IfcArbitraryProfileDefWithVoids: 18
  IfcAxis2Placement2D: 397
  IfcAxis2Placement3D: 1279
  IfcBeam: 8
  IfcBooleanClippingResult: 7
  IfcBuilding: 1
  IfcBuildingStorey: 4
  IfcCartesianPoint: 8520
  IfcCartesianTransformationOperator3D: 167
  IfcCircle: 96
  IfcCircleProfileDef: 8
  IfcColourRgb: 43
  IfcCompositeCurve: 44
  IfcCompositeCurveSegment: 322
  IfcConnectedFaceSet: 235
  IfcConnectionSurfaceGeometry: 265
  IfcConversionBasedUnit: 1
  IfcCovering: 13
  IfcCurveBoundedPlane: 81
  IfcCurveStyle: 31
  IfcCurveStyleFont: 19
  IfcCurveStyleFontPattern: 57
  IfcDimensionalExponents: 1
  IfcDirection: 134
  IfcDoor: 14
  IfcDoorLiningProperties: 6
  IfcDoorStyle: 6
  IfcDraughtingPreDefinedCurveFont: 1
  IfcElementQuantity: 21
  IfcExtrudedAreaSolid: 421
  IfcFace: 4486
  IfcFaceBasedSurfaceModel: 40
  IfcFaceBound: 60
 

## Example: Extract and Display All IFC Spaces

An **IfcSpace** represents a room or area in the building. Each space has properties like name, area, and volume. Here's how to extract them:

In [3]:
# Get all spaces
spaces = ifc.by_type("IfcSpace")

print(f"Found {len(spaces)} spaces in the building\n")
print("=" * 60)

for i, space in enumerate(spaces, 1):
    # Extract properties
    name = space.Name if space.Name else "Unnamed"
    
    # Try to get area and volume
    area = None
    volume = None
    
    if hasattr(space, 'Quantities') and space.Quantities:
        for quantity in space.Quantities.Quantities:
            if hasattr(quantity, 'Name'):
                if quantity.Name == "NetFloorArea" and hasattr(quantity, 'AreaValue'):
                    area = quantity.AreaValue
                elif quantity.Name == "GrossVolume" and hasattr(quantity, 'VolumeValue'):
                    volume = quantity.VolumeValue
    
    # Format output
    print(f"{i}. {name}")
    if area:
        print(f"   Area: {area:.2f} m²")
    if volume:
        print(f"   Volume: {volume:.2f} m³")
    print()

print("=" * 60)

Found 21 spaces in the building

1. A101

2. A201

3. B104

4. B101

5. B201

6. A205

7. B205

8. A105

9. B105

10. R301

11. B102

12. A102

13. A103

14. B103

15. B204

16. A204

17. A104

18. B203

19. A202

20. B202

21. A203



## Exercise 1: Building Code Compliance Checker (Catalonia)

**Objective:** Write a function that validates spaces against Catalonia building code requirements.

**Reference:** [Catalan Building Code](https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e)

### Key Requirements to Check

Based on the Catalan building code, residential spaces should meet these minimum standards:

| Space Type | Min Height | Min Area | Purpose |
|---|---|---|---|
| Living Room | 2.6 m | 16 m² | Main living area |
| Bedroom | 2.6 m | 9 m² | Single occupancy |
| Kitchen | 2.6 m | 8 m² | Cooking/dining |
| Bathroom | 2.3 m | 4 m² | Hygiene |
| Corridor | 2.3 m | 1.5 m² | Circulation |

### Your Task

Write a function `check_space_compliance(spaces)` that:

1. **Identifies** each space type (you can infer from the name or classify them)
2. **Extracts** the height and area properties from each space
3. **Compares** against the requirements above
4. **Reports** which spaces pass/fail and why
5. **Returns** a summary with compliance status

**Hints:**
- Height can be extracted from the Z-coordinate range of the space geometry
- Area is usually stored in space.Quantities as "NetFloorArea"
- Space names often indicate their type (e.g., "Master Bedroom", "Kitchen")

### Starter Code

In [ ]:
import json
import ifcopenshell


def check_space_compliance(spaces):
    """Validate IfcSpace entities against Catalan residential minima and return a structured report."""

    requirements = {
        "Living Room": {"min_height": 2.6, "min_area": 16.0},
        "Bedroom": {"min_height": 2.6, "min_area": 9.0},
        "Kitchen": {"min_height": 2.6, "min_area": 8.0},
        "Bathroom": {"min_height": 2.3, "min_area": 4.0},
        "Corridor": {"min_height": 2.3, "min_area": 1.5},
    }

    type_keywords = {
        "Living Room": ["living", "salon", "sala", "menjador"],
        "Bedroom": ["bedroom", "dorm", "master", "guest room", "habitacio"],
        "Kitchen": ["kitchen", "cocina", "cuina"],
        "Bathroom": ["bath", "toilet", "wc", "lavabo", "bano", "bany"],
        "Corridor": ["corridor", "hall", "hallway", "passage", "pasillo", "distribuidor"],
    }

    def _classify_space(space):
        text = " ".join(
            str(v).lower()
            for v in [getattr(space, "Name", ""), getattr(space, "LongName", "")]
            if v
        )
        for space_type, keywords in type_keywords.items():
            if any(k in text for k in keywords):
                return space_type
        return None

    def _extract_area(space):
        for rel in getattr(space, "IsDefinedBy", []) or []:
            prop_def = getattr(rel, "RelatingPropertyDefinition", None)
            if not prop_def or not prop_def.is_a("IfcElementQuantity"):
                continue
            for q in getattr(prop_def, "Quantities", []) or []:
                if q.is_a("IfcQuantityArea"):
                    qn = (getattr(q, "Name", "") or "").lower()
                    if "netfloorarea" in qn or "area" in qn:
                        return float(q.AreaValue)
        return None

    def _extract_height(space):
        for rel in getattr(space, "IsDefinedBy", []) or []:
            prop_def = getattr(rel, "RelatingPropertyDefinition", None)
            if not prop_def or not prop_def.is_a("IfcElementQuantity"):
                continue
            for q in getattr(prop_def, "Quantities", []) or []:
                if q.is_a("IfcQuantityLength"):
                    qn = (getattr(q, "Name", "") or "").lower()
                    if "height" in qn:
                        return float(q.LengthValue)

        rep = getattr(space, "Representation", None)
        if not rep:
            return None

        heights = []
        for shape_rep in getattr(rep, "Representations", []) or []:
            for item in getattr(shape_rep, "Items", []) or []:
                if item.is_a("IfcExtrudedAreaSolid"):
                    try:
                        heights.append(float(item.Depth))
                    except Exception:
                        pass
        return max(heights) if heights else None

    report = {
        "requirements": requirements,
        "spaces": [],
        "passed": [],
        "failed": [],
        "warnings": [],
        "summary": {},
        "final_report": {},
    }

    passed_checks_total = 0
    failed_checks_total = 0

    for space in spaces:
        space_id = int(space.id())
        name = getattr(space, "LongName", None) or getattr(space, "Name", None) or "Unnamed"
        space_type = _classify_space(space)
        area = _extract_area(space)
        height = _extract_height(space)

        checks = []

        # Check 1: classification
        if space_type:
            checks.append({
                "check": "space_type_classification",
                "status": "PASS",
                "reason": f"Classified as '{space_type}' from Name/LongName."
            })
        else:
            checks.append({
                "check": "space_type_classification",
                "status": "FAIL",
                "reason": "Could not classify space type from Name/LongName."
            })

        required_area = requirements.get(space_type, {}).get("min_area")
        required_height = requirements.get(space_type, {}).get("min_height")

        # Check 2: area
        if space_type is None:
            checks.append({
                "check": "minimum_area",
                "status": "FAIL",
                "reason": "Area rule cannot be evaluated because space type is unknown.",
                "actual": area,
                "required_min": None
            })
        elif area is None:
            checks.append({
                "check": "minimum_area",
                "status": "FAIL",
                "reason": "Area not found in quantity sets.",
                "actual": None,
                "required_min": required_area
            })
        elif area >= required_area:
            checks.append({
                "check": "minimum_area",
                "status": "PASS",
                "reason": f"Area {area:.2f} m2 meets minimum {required_area:.2f} m2.",
                "actual": area,
                "required_min": required_area
            })
        else:
            checks.append({
                "check": "minimum_area",
                "status": "FAIL",
                "reason": f"Area {area:.2f} m2 is below minimum {required_area:.2f} m2.",
                "actual": area,
                "required_min": required_area
            })

        # Check 3: height
        if space_type is None:
            checks.append({
                "check": "minimum_height",
                "status": "FAIL",
                "reason": "Height rule cannot be evaluated because space type is unknown.",
                "actual": height,
                "required_min": None
            })
        elif height is None:
            checks.append({
                "check": "minimum_height",
                "status": "FAIL",
                "reason": "Height not found in quantity sets or geometry.",
                "actual": None,
                "required_min": required_height
            })
        elif height >= required_height:
            checks.append({
                "check": "minimum_height",
                "status": "PASS",
                "reason": f"Height {height:.2f} m meets minimum {required_height:.2f} m.",
                "actual": height,
                "required_min": required_height
            })
        else:
            checks.append({
                "check": "minimum_height",
                "status": "FAIL",
                "reason": f"Height {height:.2f} m is below minimum {required_height:.2f} m.",
                "actual": height,
                "required_min": required_height
            })

        pass_count = sum(1 for c in checks if c["status"] == "PASS")
        fail_count = sum(1 for c in checks if c["status"] == "FAIL")
        passed_checks_total += pass_count
        failed_checks_total += fail_count

        space_report = {
            "space_id": space_id,
            "name": name,
            "space_type": space_type,
            "measurements": {
                "area_m2": area,
                "height_m": height,
            },
            "requirements_applied": {
                "min_area_m2": required_area,
                "min_height_m": required_height,
            },
            "checks": checks,
            "check_counts": {"passed": pass_count, "failed": fail_count},
            "compliant": fail_count == 0,
        }

        report["spaces"].append(space_report)

        if space_report["compliant"]:
            report["passed"].append(space_report)
        else:
            report["failed"].append(space_report)

        if space_type is None:
            report["warnings"].append({
                "space_id": space_id,
                "name": name,
                "reason": "Space type unknown; requirements could not be fully evaluated."
            })

    checked_spaces = len(report["spaces"])
    report["summary"] = {
        "checked_spaces": checked_spaces,
        "passed_spaces": len(report["passed"]),
        "failed_spaces": len(report["failed"]),
        "passed_checks_total": passed_checks_total,
        "failed_checks_total": failed_checks_total,
        "warning_count": len(report["warnings"]),
        "overall_compliance": checked_spaces > 0 and len(report["failed"]) == 0,
    }

    report["final_report"] = {
        "summary": report["summary"],
        "passed_spaces_details": report["passed"],
        "failed_spaces_details": report["failed"],
        "warnings": report["warnings"],
    }

    return report


# Run
ifc = ifcopenshell.open("assets/duplex.ifc")
spaces = ifc.by_type("IfcSpace")
result = check_space_compliance(spaces)

# Structured final report output
print(json.dumps(result["final_report"], indent=2))


Summary: {'checked_spaces': 14, 'passed_count': 2, 'failed_count': 12, 'warning_count': 7, 'compliance': False}
Passed: 2
Failed: 12
Warnings: 7

First failed spaces:
- Hallway: Height not found in quantity sets or geometry.
- Bathroom 1: Area 4.00 m2 < minimum 4.00 m2
- Hallway: Height not found in quantity sets or geometry.
- Living Room: Height 2.58 m < minimum 2.60 m
- Living Room: Height 2.58 m < minimum 2.60 m


## Exercise 2: Window Detection and Compliance Verification

**Objective:** Find all windows in the model and verify they meet natural light and ventilation requirements.

**Reference:** [Catalan Building Code](https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e)

### Key Requirements

Residential spaces must have:
- **Minimum window area** = 1/8 of floor area (12.5%)
- **Minimum window dimensions** = 60cm width, 100cm height (for single opening)
- Living areas should have direct natural light

### Your Task

Write a function `analyze_window_compliance(ifc_model, spaces)` that:

1. **Finds all IfcWindow** entities in the model
2. **Links windows to spaces** (which room is each window in?)
3. **Extracts properties**: dimensions, area, orientation
4. **Calculates window-to-floor ratio** for each space
5. **Reports compliance** for each space with windows

**Hints:**
- Windows are `IfcWindow` entities
- To link windows to spaces, check spatial containment relationships
- Window area can be calculated from the frame/pane dimensions
- You may need to examine the geometric representation

### Starter Code

In [15]:
import json
import math
import ifcopenshell


def analyze_window_compliance(ifc_model, spaces):
    """
    Analyze windows in each space and return a structured pass/fail compliance report.
    """

    requirements = {
        "min_window_to_floor_ratio": 0.125,   # 1/8
        "min_window_width_m": 0.60,
        "min_window_height_m": 1.00,
        "living_area_requires_direct_light": True,
    }

    windows = ifc_model.by_type("IfcWindow")

    report = {
        "requirements": requirements,
        "total_windows": len(windows),
        "windows": [],
        "windows_by_space": {},
        "space_results": {},
        "checks": [],
        "passed_checks": [],
        "failed_checks": [],
        "warnings": [],
        "passed_spaces": [],
        "failed_spaces": [],
        "summary": {},
        "compliance": False,
        "final_report": {},
    }

    def _add_check(check_name, status, reason, scope="global", entity_id=None, entity_name=None, details=None):
        row = {
            "check": check_name,
            "status": status,  # PASS | FAIL | WARN
            "reason": reason,
            "scope": scope,
            "entity_id": entity_id,
            "entity_name": entity_name,
            "details": details or {},
        }
        report["checks"].append(row)
        if status == "PASS":
            report["passed_checks"].append(row)
        elif status == "FAIL":
            report["failed_checks"].append(row)
        else:
            report["warnings"].append(row)

    def _safe_float(v):
        try:
            return float(v)
        except Exception:
            return None

    def _space_name(space):
        return getattr(space, "LongName", None) or getattr(space, "Name", None) or f"Space_{space.id()}"

    def _is_living_area(space):
        text = f"{getattr(space, 'Name', '')} {getattr(space, 'LongName', '')}".lower()
        keywords = ["living", "salon", "sala", "menjador", "estar"]
        return any(k in text for k in keywords)

    def _extract_space_area(space):
        for rel in getattr(space, "IsDefinedBy", []) or []:
            prop_def = getattr(rel, "RelatingPropertyDefinition", None)
            if not prop_def or not prop_def.is_a("IfcElementQuantity"):
                continue
            for q in getattr(prop_def, "Quantities", []) or []:
                if q.is_a("IfcQuantityArea"):
                    qn = (getattr(q, "Name", "") or "").lower()
                    if "netfloorarea" in qn or "floorarea" in qn or "area" in qn:
                        return _safe_float(getattr(q, "AreaValue", None))
        return None

    def _extract_window_dimensions(window):
        width = _safe_float(getattr(window, "OverallWidth", None))
        height = _safe_float(getattr(window, "OverallHeight", None))

        if width is not None and height is not None:
            return width, height

        for rel in getattr(window, "IsDefinedBy", []) or []:
            prop_def = getattr(rel, "RelatingPropertyDefinition", None)
            if not prop_def or not prop_def.is_a("IfcElementQuantity"):
                continue
            for q in getattr(prop_def, "Quantities", []) or []:
                if not q.is_a("IfcQuantityLength"):
                    continue
                qn = (getattr(q, "Name", "") or "").lower()
                val = _safe_float(getattr(q, "LengthValue", None))
                if val is None:
                    continue
                if width is None and "width" in qn:
                    width = val
                if height is None and "height" in qn:
                    height = val

        return width, height

    def _extract_window_area(window, width, height):
        if width is not None and height is not None:
            return width * height

        for rel in getattr(window, "IsDefinedBy", []) or []:
            prop_def = getattr(rel, "RelatingPropertyDefinition", None)
            if not prop_def or not prop_def.is_a("IfcElementQuantity"):
                continue
            for q in getattr(prop_def, "Quantities", []) or []:
                if q.is_a("IfcQuantityArea"):
                    return _safe_float(getattr(q, "AreaValue", None))
        return None

    def _vector_to_cardinal(x, y):
        angle = (math.degrees(math.atan2(y, x)) + 360.0) % 360.0
        dirs = [("E", 0.0), ("NE", 45.0), ("N", 90.0), ("NW", 135.0), ("W", 180.0), ("SW", 225.0), ("S", 270.0), ("SE", 315.0)]
        best = min(dirs, key=lambda d: min(abs(angle - d[1]), 360.0 - abs(angle - d[1])))
        return best[0]

    def _extract_orientation(window):
        placement = getattr(window, "ObjectPlacement", None)
        while placement and placement.is_a("IfcLocalPlacement"):
            rp = getattr(placement, "RelativePlacement", None)
            ref = getattr(rp, "RefDirection", None) if rp else None
            if ref and len(getattr(ref, "DirectionRatios", [])) >= 2:
                ratios = ref.DirectionRatios
                x = _safe_float(ratios[0]) or 0.0
                y = _safe_float(ratios[1]) or 0.0
                if abs(x) > 1e-9 or abs(y) > 1e-9:
                    return _vector_to_cardinal(x, y)
            placement = getattr(placement, "PlacementRelTo", None)

        text = f"{getattr(window, 'Name', '')} {getattr(window, 'ObjectType', '')}".lower()
        for k, v in {"north": "N", "south": "S", "east": "E", "west": "W", "norte": "N", "sud": "S", "est": "E", "oest": "W"}.items():
            if k in text:
                return v
        return "Unknown"

    # Build ID maps
    space_by_id = {int(s.id()): s for s in spaces}
    window_by_id = {int(w.id()): w for w in windows}
    space_window_ids = {sid: set() for sid in space_by_id.keys()}

    # Link windows to spaces: IfcRelSpaceBoundary
    for rel in ifc_model.by_type("IfcRelSpaceBoundary"):
        sp = getattr(rel, "RelatingSpace", None)
        elem = getattr(rel, "RelatedBuildingElement", None)
        if sp and elem and elem.is_a("IfcWindow"):
            sid = int(sp.id())
            wid = int(elem.id())
            if sid in space_window_ids:
                space_window_ids[sid].add(wid)

    # Fallback links
    for w in windows:
        wid = int(w.id())
        for rel in getattr(w, "ProvidesBoundaries", []) or []:
            sp = getattr(rel, "RelatingSpace", None)
            if sp is not None and int(sp.id()) in space_window_ids:
                space_window_ids[int(sp.id())].add(wid)
        for rel in getattr(w, "ContainedInStructure", []) or []:
            struct = getattr(rel, "RelatingStructure", None)
            if struct is not None and struct.is_a("IfcSpace") and int(struct.id()) in space_window_ids:
                space_window_ids[int(struct.id())].add(wid)

    linked_count = sum(len(v) for v in space_window_ids.values())
    if linked_count > 0:
        _add_check("window_space_linkage", "PASS", f"Created {linked_count} space-window link(s).")
    else:
        _add_check("window_space_linkage", "FAIL", "No windows could be linked to spaces.")

    # Window-level results
    for wid, w in window_by_id.items():
        width, height = _extract_window_dimensions(w)
        area = _extract_window_area(w, width, height)
        orientation = _extract_orientation(w)
        linked_space_ids = sorted([sid for sid, wids in space_window_ids.items() if wid in wids])

        if width is None or height is None:
            dim_status = "WARN"
            dim_reason = "Window dimensions are incomplete; min opening check could not be fully evaluated."
            _add_check("window_min_dimension", "WARN", f"Window {wid}: {dim_reason}", scope="window", entity_id=wid, entity_name=getattr(w, "Name", None))
        elif width >= requirements["min_window_width_m"] and height >= requirements["min_window_height_m"]:
            dim_status = "PASS"
            dim_reason = f"Width {width:.2f} m and height {height:.2f} m meet minimum dimensions."
            _add_check("window_min_dimension", "PASS", f"Window {wid}: {dim_reason}", scope="window", entity_id=wid, entity_name=getattr(w, "Name", None))
        else:
            dim_status = "FAIL"
            dim_reason = (
                f"Width/height ({width:.2f} m, {height:.2f} m) below minimum "
                f"({requirements['min_window_width_m']:.2f} m, {requirements['min_window_height_m']:.2f} m)."
            )
            _add_check("window_min_dimension", "FAIL", f"Window {wid}: {dim_reason}", scope="window", entity_id=wid, entity_name=getattr(w, "Name", None))

        report["windows"].append({
            "window_id": wid,
            "global_id": getattr(w, "GlobalId", None),
            "name": getattr(w, "Name", None) or f"Window_{wid}",
            "width_m": width,
            "height_m": height,
            "area_m2": area,
            "orientation": orientation,
            "space_ids": linked_space_ids,
            "dimension_check": {"status": dim_status, "reason": dim_reason},
        })

    # windows_by_space
    for sid, wids in space_window_ids.items():
        if wids:
            report["windows_by_space"][f"{_space_name(space_by_id[sid])} ({sid})"] = sorted(list(wids))

    # Space-level compliance
    for sid, s in space_by_id.items():
        sname = _space_name(s)
        floor_area = _extract_space_area(s)
        linked_windows = [row for row in report["windows"] if sid in row["space_ids"]]
        window_count = len(linked_windows)

        total_window_area = sum(w["area_m2"] for w in linked_windows if w["area_m2"] is not None)
        min_required_window_area = (floor_area * requirements["min_window_to_floor_ratio"]) if floor_area is not None else None
        ratio = (total_window_area / floor_area) if (floor_area not in (None, 0)) else None
        has_min_dimension_window = any(w["dimension_check"]["status"] == "PASS" for w in linked_windows)
        living_area = _is_living_area(s)

        checks = []

        # Check 1: floor area availability
        if floor_area is None:
            checks.append({"check": "floor_area_available", "status": "FAIL", "reason": "Floor area not found in quantity sets."})
        else:
            checks.append({"check": "floor_area_available", "status": "PASS", "reason": f"Floor area available: {floor_area:.2f} m2."})

        # Check 2: minimum total window area ratio
        if floor_area is None:
            checks.append({"check": "window_to_floor_ratio", "status": "FAIL", "reason": "Cannot evaluate ratio without floor area."})
        elif ratio is None:
            checks.append({"check": "window_to_floor_ratio", "status": "FAIL", "reason": "Cannot compute ratio due to missing window area data."})
        elif ratio >= requirements["min_window_to_floor_ratio"]:
            checks.append({
                "check": "window_to_floor_ratio",
                "status": "PASS",
                "reason": f"Ratio {ratio:.3f} meets minimum {requirements['min_window_to_floor_ratio']:.3f}.",
            })
        else:
            checks.append({
                "check": "window_to_floor_ratio",
                "status": "FAIL",
                "reason": f"Ratio {ratio:.3f} is below minimum {requirements['min_window_to_floor_ratio']:.3f}.",
            })

        # Check 3: minimum opening dimensions present
        if window_count == 0:
            checks.append({"check": "minimum_opening_dimension", "status": "FAIL", "reason": "No windows linked to this space."})
        elif has_min_dimension_window:
            checks.append({
                "check": "minimum_opening_dimension",
                "status": "PASS",
                "reason": "At least one linked window meets minimum dimensions (0.60m x 1.00m).",
            })
        else:
            checks.append({
                "check": "minimum_opening_dimension",
                "status": "FAIL",
                "reason": "Linked windows exist but none meet minimum dimensions (0.60m x 1.00m).",
            })

        # Check 4: living space direct natural light
        if living_area:
            if window_count > 0:
                checks.append({"check": "living_space_direct_natural_light", "status": "PASS", "reason": "Living area has at least one linked window."})
            else:
                checks.append({"check": "living_space_direct_natural_light", "status": "FAIL", "reason": "Living area has no linked windows."})
        else:
            checks.append({"check": "living_space_direct_natural_light", "status": "PASS", "reason": "Not a living area; rule not mandatory for this space type."})

        for c in checks:
            _add_check(
                c["check"],
                c["status"],
                f"{sname} ({sid}): {c['reason']}",
                scope="space",
                entity_id=sid,
                entity_name=sname,
            )

        failed_in_space = [c for c in checks if c["status"] == "FAIL"]
        space_compliant = len(failed_in_space) == 0

        space_row = {
            "space_id": sid,
            "space_name": sname,
            "is_living_area": living_area,
            "floor_area_m2": floor_area,
            "window_count": window_count,
            "total_window_area_m2": total_window_area,
            "minimum_required_window_area_m2": min_required_window_area,
            "window_to_floor_ratio": ratio,
            "has_min_dimension_window": has_min_dimension_window,
            "checks": checks,
            "compliant": space_compliant,
        }

        report["space_results"][str(sid)] = space_row
        if space_compliant:
            report["passed_spaces"].append(space_row)
        else:
            report["failed_spaces"].append(space_row)

    report["compliance"] = len(report["failed_checks"]) == 0
    report["summary"] = {
        "total_spaces_checked": len(space_by_id),
        "spaces_with_windows": sum(1 for sid in space_by_id if len([w for w in report["windows"] if sid in w["space_ids"]]) > 0),
        "passed_spaces": len(report["passed_spaces"]),
        "failed_spaces": len(report["failed_spaces"]),
        "total_checks": len(report["checks"]),
        "passed_checks": len(report["passed_checks"]),
        "failed_checks": len(report["failed_checks"]),
        "warning_checks": len(report["warnings"]),
        "overall_compliance": report["compliance"],
    }

    report["final_report"] = {
        "summary": report["summary"],
        "passed_checks_details": report["passed_checks"],
        "failed_checks_details": report["failed_checks"],
        "warnings": report["warnings"],
        "passed_spaces_details": report["passed_spaces"],
        "failed_spaces_details": report["failed_spaces"],
        "window_details": report["windows"],
    }

    return report


# Standalone run block
ifc = ifcopenshell.open("assets/duplex.ifc")
spaces = ifc.by_type("IfcSpace")
analysis = analyze_window_compliance(ifc, spaces)

# Structured final report output
print(json.dumps(analysis["final_report"], indent=2))


{
  "summary": {
    "total_spaces_checked": 21,
    "spaces_with_windows": 8,
    "passed_spaces": 6,
    "failed_spaces": 15,
    "total_checks": 109,
    "passed_checks": 67,
    "failed_checks": 42,
    "warning_checks": 0,
    "overall_compliance": false
  },
  "passed_checks_details": [
    {
      "check": "window_space_linkage",
      "status": "PASS",
      "reason": "Created 16 space-window link(s).",
      "scope": "global",
      "entity_id": null,
      "entity_name": null,
      "details": {}
    },
    {
      "check": "window_min_dimension",
      "status": "PASS",
      "reason": "Window 6426: Width 4.83 m and height 2.42 m meet minimum dimensions.",
      "scope": "window",
      "entity_id": 6426,
      "entity_name": "M_Fixed:4835mm x 2420mm:4835mm x 2420mm:145788",
      "details": {}
    },
    {
      "check": "window_min_dimension",
      "status": "PASS",
      "reason": "Window 6531: Width 4.83 m and height 2.42 m meet minimum dimensions.",
      "scope": "win

## Bonus Exercise: Fire Safety Route Analysis

**Objective:** Find the longest evacuation route within the apartment and verify it meets fire safety requirements.

**Difficulty:** Advanced

**Reference:** [Catalan Building Code - Fire Safety Section](https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e)

### Fire Safety Requirements

According to the Catalan building code:
- **Maximum travel distance** to exit: ≤ 25-30 m (depending on building type)
- **Minimum corridor width**: 1.2 m
- **Minimum door width**: 0.8 m (for exits)
- **No dead-end corridors** longer than 10 m

### Your Task

Write a function `analyze_evacuation_routes(ifc_model, spaces)` that:

1. **Builds a spatial graph** of the apartment (rooms as nodes, doors/openings as connections)
2. **Calculates distances** between spaces (using area/perimeter as proxy)
3. **Finds the longest route** from any point to the nearest exit
4. **Validates** the route against fire safety requirements
5. **Identifies bottlenecks** (narrow corridors, small doors)

**Hints:**
- Think of spaces as nodes and doors as edges in a graph
- Use BFS/DFS to find longest paths
- Door dimensions can indicate width constraints
- Consider calculating distances based on space geometry
- This is a simplified model - real analysis would use detailed geometry

### Starter Code

```python
def analyze_evacuation_routes(ifc_model, spaces):
    """
    Analyze evacuation routes and fire safety compliance.
    
    Args:
        ifc_model: The loaded IFC model
        spaces: List of IfcSpace objects
        
    Returns:
        dict: Fire safety analysis and recommendations
    """
    
    # Get all doors (potential connections between spaces)
    doors = ifc_model.by_type("IfcDoor")
    
    analysis = {
        "total_spaces": len(spaces),
        "total_doors": len(doors),
        "longest_route": None,
        "longest_distance": 0,
        "safety_issues": [],
        "compliant": False
    }
    
    print(f"Analyzing {len(spaces)} spaces with {len(doors)} doors")
    
    # TODO: Build spatial connectivity graph
    # TODO: Find longest evacuation path
    # TODO: Check corridor widths and door dimensions
    # TODO: Validate against requirements
    # TODO: Report issues and recommendations
    
    return analysis


# Run the analysis (uncomment when ready)
# fire_analysis = analyze_evacuation_routes(ifc, spaces)
```

In [14]:
import json
from collections import defaultdict
import heapq
import itertools
import math
import ifcopenshell


def analyze_evacuation_routes(ifc_model, spaces):
    """
    Analyze evacuation routes and fire safety compliance with structured pass/fail reporting.
    """

    requirements = {
        "max_travel_distance_m": 25.0,
        "min_corridor_width_m": 1.2,
        "min_exit_door_width_m": 0.8,
        "max_dead_end_length_m": 10.0,
    }

    doors = ifc_model.by_type("IfcDoor")
    space_by_id = {int(s.id()): s for s in spaces}

    analysis = {
        "requirements": requirements,
        "total_spaces": len(spaces),
        "total_doors": len(doors),
        "exit_space_ids": [],
        "space_graph_nodes": len(space_by_id),
        "space_graph_edges": 0,
        "longest_route": None,
        "longest_distance": 0.0,
        "door_results": [],
        "corridor_results": [],
        "checks": [],
        "passed_checks": [],
        "failed_checks": [],
        "warnings": [],
        "safety_issues": [],
        "compliant": False,
        "summary": {},
        "final_report": {},
    }

    def _add_check(check_name, status, reason, scope="global", entity_id=None, entity_name=None, details=None):
        row = {
            "check": check_name,
            "status": status,  # PASS | FAIL | WARN
            "reason": reason,
            "scope": scope,
            "entity_id": entity_id,
            "entity_name": entity_name,
            "details": details or {},
        }
        analysis["checks"].append(row)
        if status == "PASS":
            analysis["passed_checks"].append(row)
        elif status == "FAIL":
            analysis["failed_checks"].append(row)
            analysis["safety_issues"].append(reason)
        else:
            analysis["warnings"].append(row)

    # ---------- Helpers ----------
    def _safe_float(value):
        try:
            return float(value)
        except Exception:
            return None

    def _space_name(space):
        return getattr(space, "LongName", None) or getattr(space, "Name", None) or f"Space_{space.id()}"

    def _entity_quantities(entity):
        for rel in getattr(entity, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcElementQuantity"):
                for q in getattr(pdef, "Quantities", []) or []:
                    yield q

    def _extract_area(entity):
        for q in _entity_quantities(entity):
            if q.is_a("IfcQuantityArea"):
                qn = (getattr(q, "Name", "") or "").lower()
                if "area" in qn or "netfloorarea" in qn:
                    return _safe_float(getattr(q, "AreaValue", None))
        return None

    def _extract_length(entity, name_keywords):
        for q in _entity_quantities(entity):
            if q.is_a("IfcQuantityLength"):
                qn = (getattr(q, "Name", "") or "").lower()
                if any(k in qn for k in name_keywords):
                    return _safe_float(getattr(q, "LengthValue", None))
        return None

    def _extract_door_width(door):
        width = _safe_float(getattr(door, "OverallWidth", None))
        if width is not None:
            return width
        return _extract_length(door, ["width", "clear width", "nominal width"])

    def _extract_is_external(door):
        for rel in getattr(door, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if not pdef or not pdef.is_a("IfcPropertySet"):
                continue
            for prop in getattr(pdef, "HasProperties", []) or []:
                if not prop.is_a("IfcPropertySingleValue"):
                    continue
                if (getattr(prop, "Name", "") or "").lower() != "isexternal":
                    continue
                nv = getattr(prop, "NominalValue", None)
                if nv is None:
                    return None
                txt = str(nv).strip().lower()
                if txt in {"true", ".t.", "t", "1"}:
                    return True
                if txt in {"false", ".f.", "f", "0"}:
                    return False
                wrapped = getattr(nv, "wrappedValue", None)
                if isinstance(wrapped, bool):
                    return wrapped
        return None

    def _door_name_flags(door):
        text = f"{getattr(door, 'Name', '')} {getattr(door, 'LongName', '')} {getattr(door, 'ObjectType', '')}".lower()
        exit_words = ["exit", "entrance", "entry", "main door", "front door", "outside", "egress"]
        return any(w in text for w in exit_words)

    def _is_corridor(space):
        text = f"{getattr(space, 'Name', '')} {getattr(space, 'LongName', '')}".lower()
        keys = ["corridor", "hall", "hallway", "pasillo", "distribuidor", "passage"]
        return any(k in text for k in keys)

    def _edge_distance(space_id_a, space_id_b):
        a1 = _extract_area(space_by_id[space_id_a])
        a2 = _extract_area(space_by_id[space_id_b])
        if a1 is not None and a2 is not None:
            return 0.5 * (math.sqrt(max(a1, 0.0)) + math.sqrt(max(a2, 0.0)))
        if a1 is not None:
            return math.sqrt(max(a1, 0.0))
        if a2 is not None:
            return math.sqrt(max(a2, 0.0))
        return 2.0

    # ---------- Map doors to spaces ----------
    door_to_spaces = defaultdict(set)

    for rel in ifc_model.by_type("IfcRelSpaceBoundary"):
        sp = getattr(rel, "RelatingSpace", None)
        elem = getattr(rel, "RelatedBuildingElement", None)
        if sp is not None and elem is not None and elem.is_a("IfcDoor"):
            door_to_spaces[int(elem.id())].add(int(sp.id()))

    for door in doors:
        did = int(door.id())
        for rel in getattr(door, "ProvidesBoundaries", []) or []:
            sp = getattr(rel, "RelatingSpace", None)
            if sp is not None:
                door_to_spaces[did].add(int(sp.id()))

    # ---------- Build graph ----------
    adjacency = defaultdict(list)
    edge_counter = 0
    door_rows = []
    exit_space_ids = set()

    for door in doors:
        did = int(door.id())
        linked_spaces = sorted(sid for sid in door_to_spaces.get(did, set()) if sid in space_by_id)

        width = _extract_door_width(door)
        is_external = _extract_is_external(door)
        name_suggests_exit = _door_name_flags(door)
        is_exit_door = (is_external is True) or name_suggests_exit or (len(linked_spaces) == 1)

        if is_exit_door:
            for sid in linked_spaces:
                exit_space_ids.add(sid)

        if len(linked_spaces) >= 2:
            for a, b in itertools.combinations(linked_spaces, 2):
                d = _edge_distance(a, b)
                adjacency[a].append((b, d, did))
                adjacency[b].append((a, d, did))
                edge_counter += 1

        door_rows.append({
            "door_id": did,
            "name": getattr(door, "Name", None) or f"Door_{did}",
            "width_m": width,
            "linked_space_ids": linked_spaces,
            "is_exit_door": is_exit_door,
            "is_external": is_external,
        })

    if not exit_space_ids and space_by_id:
        fallback_sid = next(iter(space_by_id.keys()))
        exit_space_ids.add(fallback_sid)
        _add_check(
            "exit_identification",
            "WARN",
            f"No explicit exit door found. Used fallback exit space {_space_name(space_by_id[fallback_sid])} ({fallback_sid}).",
            scope="global",
        )
    else:
        _add_check(
            "exit_identification",
            "PASS",
            f"Identified {len(exit_space_ids)} exit-connected space(s).",
            scope="global",
        )

    analysis["exit_space_ids"] = sorted(exit_space_ids)
    analysis["space_graph_edges"] = edge_counter

    if edge_counter > 0:
        _add_check("graph_connectivity_build", "PASS", f"Spatial graph built with {edge_counter} edge(s).")
    else:
        _add_check("graph_connectivity_build", "WARN", "Spatial graph has 0 edges; route analysis may be limited.")

    # ---------- Multi-source Dijkstra ----------
    INF = float("inf")
    dist = {sid: INF for sid in space_by_id}
    prev = {}
    prev_door = {}
    pq = []

    for sid in exit_space_ids:
        if sid in dist:
            dist[sid] = 0.0
            heapq.heappush(pq, (0.0, sid))

    while pq:
        cur_d, cur = heapq.heappop(pq)
        if cur_d > dist[cur]:
            continue
        for nxt, w, door_id in adjacency.get(cur, []):
            nd = cur_d + w
            if nd < dist[nxt]:
                dist[nxt] = nd
                prev[nxt] = cur
                prev_door[nxt] = door_id
                heapq.heappush(pq, (nd, nxt))

    reachable = [sid for sid, d in dist.items() if d < INF]
    unreachable = [sid for sid, d in dist.items() if d == INF]

    if unreachable:
        names = [f"{_space_name(space_by_id[sid])} ({sid})" for sid in unreachable]
        _add_check(
            "all_spaces_reachable_to_exit",
            "FAIL",
            "Some spaces are not connected to an exit path.",
            details={"unreachable_spaces": names},
        )
    else:
        _add_check("all_spaces_reachable_to_exit", "PASS", "All spaces are connected to at least one exit path.")

    if reachable:
        worst_sid = max(reachable, key=lambda sid: dist[sid])
        worst_dist = dist[worst_sid]
        analysis["longest_distance"] = worst_dist

        path_space_ids = [worst_sid]
        route_door_ids = []
        cur = worst_sid
        while cur in prev:
            route_door_ids.append(prev_door[cur])
            cur = prev[cur]
            path_space_ids.append(cur)

        path_space_names = [_space_name(space_by_id[sid]) for sid in path_space_ids]
        analysis["longest_route"] = {
            "from_space_id": worst_sid,
            "from_space_name": _space_name(space_by_id[worst_sid]),
            "to_exit_space_id": path_space_ids[-1],
            "to_exit_space_name": _space_name(space_by_id[path_space_ids[-1]]),
            "path_space_ids": path_space_ids,
            "path_space_names": path_space_names,
            "route_door_ids": route_door_ids,
            "distance_m": worst_dist,
        }

        if worst_dist <= requirements["max_travel_distance_m"]:
            _add_check(
                "max_travel_distance",
                "PASS",
                f"Longest travel distance {worst_dist:.2f} m is within limit {requirements['max_travel_distance_m']:.2f} m.",
                details={"distance_m": worst_dist},
            )
        else:
            _add_check(
                "max_travel_distance",
                "FAIL",
                f"Longest travel distance {worst_dist:.2f} m exceeds limit {requirements['max_travel_distance_m']:.2f} m.",
                details={"distance_m": worst_dist},
            )
    else:
        _add_check("max_travel_distance", "FAIL", "No reachable evacuation route could be computed.")

    # ---------- Door checks ----------
    exit_door_rows = [d for d in door_rows if d["is_exit_door"]]
    if not exit_door_rows:
        _add_check("exit_door_presence", "FAIL", "No exit doors identified.")
    else:
        _add_check("exit_door_presence", "PASS", f"Found {len(exit_door_rows)} exit door candidate(s).")

    for d in exit_door_rows:
        did = d["door_id"]
        dname = d["name"]
        width = d["width_m"]

        if width is None:
            check = {
                "door_id": did,
                "door_name": dname,
                "status": "WARN",
                "reason": "Door width could not be read.",
                "width_m": None,
                "required_min_width_m": requirements["min_exit_door_width_m"],
            }
            analysis["door_results"].append(check)
            _add_check(
                "exit_door_min_width",
                "WARN",
                f"Exit door '{dname}' ({did}) width is missing.",
                scope="door",
                entity_id=did,
                entity_name=dname,
            )
        elif width >= requirements["min_exit_door_width_m"]:
            check = {
                "door_id": did,
                "door_name": dname,
                "status": "PASS",
                "reason": f"Width {width:.2f} m meets minimum {requirements['min_exit_door_width_m']:.2f} m.",
                "width_m": width,
                "required_min_width_m": requirements["min_exit_door_width_m"],
            }
            analysis["door_results"].append(check)
            _add_check(
                "exit_door_min_width",
                "PASS",
                check["reason"],
                scope="door",
                entity_id=did,
                entity_name=dname,
            )
        else:
            check = {
                "door_id": did,
                "door_name": dname,
                "status": "FAIL",
                "reason": f"Width {width:.2f} m is below minimum {requirements['min_exit_door_width_m']:.2f} m.",
                "width_m": width,
                "required_min_width_m": requirements["min_exit_door_width_m"],
            }
            analysis["door_results"].append(check)
            _add_check(
                "exit_door_min_width",
                "FAIL",
                f"Exit door '{dname}' ({did}) width {width:.2f} m is below minimum {requirements['min_exit_door_width_m']:.2f} m.",
                scope="door",
                entity_id=did,
                entity_name=dname,
            )

    # ---------- Corridor checks ----------
    corridor_count = 0
    for sid, sp in space_by_id.items():
        if not _is_corridor(sp):
            continue
        corridor_count += 1
        sname = _space_name(sp)

        corridor_width = _extract_length(sp, ["width"])
        corridor_length = _extract_length(sp, ["length"])
        if corridor_length is None:
            area = _extract_area(sp)
            if area is not None:
                corridor_length = math.sqrt(max(area, 0.0))

        degree = len(adjacency.get(sid, []))
        is_dead_end = degree <= 1 and sid not in exit_space_ids

        # Width check
        if corridor_width is None:
            w_status = "WARN"
            w_reason = "Corridor width missing; width check could not be evaluated."
            _add_check(
                "corridor_min_width",
                "WARN",
                f"Corridor '{sname}' ({sid}): {w_reason}",
                scope="corridor",
                entity_id=sid,
                entity_name=sname,
            )
        elif corridor_width >= requirements["min_corridor_width_m"]:
            w_status = "PASS"
            w_reason = f"Width {corridor_width:.2f} m meets minimum {requirements['min_corridor_width_m']:.2f} m."
            _add_check(
                "corridor_min_width",
                "PASS",
                f"Corridor '{sname}' ({sid}): {w_reason}",
                scope="corridor",
                entity_id=sid,
                entity_name=sname,
            )
        else:
            w_status = "FAIL"
            w_reason = f"Width {corridor_width:.2f} m below minimum {requirements['min_corridor_width_m']:.2f} m."
            _add_check(
                "corridor_min_width",
                "FAIL",
                f"Corridor '{sname}' ({sid}): {w_reason}",
                scope="corridor",
                entity_id=sid,
                entity_name=sname,
            )

        # Dead-end check
        if not is_dead_end:
            d_status = "PASS"
            d_reason = "Not a dead-end corridor."
            _add_check(
                "dead_end_corridor_length",
                "PASS",
                f"Corridor '{sname}' ({sid}): {d_reason}",
                scope="corridor",
                entity_id=sid,
                entity_name=sname,
            )
        else:
            if corridor_length is None:
                d_status = "WARN"
                d_reason = "Dead-end corridor length missing; dead-end length check could not be evaluated."
                _add_check(
                    "dead_end_corridor_length",
                    "WARN",
                    f"Corridor '{sname}' ({sid}): {d_reason}",
                    scope="corridor",
                    entity_id=sid,
                    entity_name=sname,
                )
            elif corridor_length <= requirements["max_dead_end_length_m"]:
                d_status = "PASS"
                d_reason = (
                    f"Dead-end length {corridor_length:.2f} m is within limit "
                    f"{requirements['max_dead_end_length_m']:.2f} m."
                )
                _add_check(
                    "dead_end_corridor_length",
                    "PASS",
                    f"Corridor '{sname}' ({sid}): {d_reason}",
                    scope="corridor",
                    entity_id=sid,
                    entity_name=sname,
                )
            else:
                d_status = "FAIL"
                d_reason = (
                    f"Dead-end length {corridor_length:.2f} m exceeds limit "
                    f"{requirements['max_dead_end_length_m']:.2f} m."
                )
                _add_check(
                    "dead_end_corridor_length",
                    "FAIL",
                    f"Corridor '{sname}' ({sid}): {d_reason}",
                    scope="corridor",
                    entity_id=sid,
                    entity_name=sname,
                )

        analysis["corridor_results"].append({
            "space_id": sid,
            "space_name": sname,
            "is_dead_end": is_dead_end,
            "width_m": corridor_width,
            "length_m": corridor_length,
            "width_check": {"status": w_status, "reason": w_reason},
            "dead_end_check": {"status": d_status, "reason": d_reason},
        })

    if corridor_count == 0:
        _add_check("corridor_presence", "WARN", "No corridors identified from space names.")
    else:
        _add_check("corridor_presence", "PASS", f"Identified {corridor_count} corridor space(s).")

    # ---------- Final summary ----------
    analysis["compliant"] = len(analysis["failed_checks"]) == 0
    analysis["summary"] = {
        "total_checks": len(analysis["checks"]),
        "passed_checks": len(analysis["passed_checks"]),
        "failed_checks": len(analysis["failed_checks"]),
        "warning_checks": len(analysis["warnings"]),
        "overall_compliance": analysis["compliant"],
        "longest_distance_m": analysis["longest_distance"],
        "total_spaces": analysis["total_spaces"],
        "total_doors": analysis["total_doors"],
    }

    analysis["final_report"] = {
        "summary": analysis["summary"],
        "longest_route": analysis["longest_route"],
        "passed_checks_details": analysis["passed_checks"],
        "failed_checks_details": analysis["failed_checks"],
        "warnings": analysis["warnings"],
        "door_results": analysis["door_results"],
        "corridor_results": analysis["corridor_results"],
    }

    return analysis


# ---------- Run block ----------
ifc = ifcopenshell.open("assets/duplex.ifc")
spaces = ifc.by_type("IfcSpace")
fire_analysis = analyze_evacuation_routes(ifc, spaces)

# Structured final report output
print(json.dumps(fire_analysis["final_report"], indent=2))


{
  "summary": {
    "total_checks": 14,
    "passed_checks": 11,
    "failed_checks": 1,
    "warning_checks": 2,
    "overall_compliance": false,
    "longest_distance_m": 3.1172786851679035,
    "total_spaces": 21,
    "total_doors": 14
  },
  "longest_route": {
    "from_space_id": 1928,
    "from_space_name": "Bathroom 1",
    "to_exit_space_id": 2108,
    "to_exit_space_name": "Foyer",
    "path_space_ids": [
      1928,
      2108
    ],
    "path_space_names": [
      "Bathroom 1",
      "Foyer"
    ],
    "route_door_ids": [
      8169
    ],
    "distance_m": 3.1172786851679035
  },
  "passed_checks_details": [
    {
      "check": "exit_identification",
      "status": "PASS",
      "reason": "Identified 4 exit-connected space(s).",
      "scope": "global",
      "entity_id": null,
      "entity_name": null,
      "details": {}
    },
    {
      "check": "graph_connectivity_build",
      "status": "PASS",
      "reason": "Spatial graph built with 12 edge(s).",
      "scope"

In [18]:
import json
import os
from datetime import datetime


def _fmt(v):
    if v is None:
        return "-"
    if isinstance(v, float):
        return f"{v:.3f}".rstrip("0").rstrip(".")
    if isinstance(v, bool):
        return "PASS" if v else "FAIL"
    if isinstance(v, (list, dict)):
        return json.dumps(v, ensure_ascii=False)
    return str(v)


def _md_escape(text):
    return str(text).replace("|", "\\|").replace("\n", "<br>")


def _md_table(headers, rows):
    if not rows:
        return "_No data_\n"
    out = []
    out.append("| " + " | ".join(headers) + " |")
    out.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for r in rows:
        out.append("| " + " | ".join(_md_escape(_fmt(c)) for c in r) + " |")
    return "\n".join(out) + "\n"


def _get(d, key, default):
    return d.get(key, default) if isinstance(d, dict) else default


def _space_fail_reasons(space_row):
    if "issues" in space_row and isinstance(space_row["issues"], list):
        return "; ".join(space_row["issues"]) if space_row["issues"] else "-"
    checks = space_row.get("checks", [])
    fails = [c.get("reason", "") for c in checks if c.get("status") == "FAIL"]
    return "; ".join(fails) if fails else "-"


def build_readable_report(data):
    e1 = _get(data, "exercise_1_space_compliance", {})
    e2 = _get(data, "exercise_2_window_compliance", {})
    e3 = _get(data, "bonus_fire_safety", {})

    lines = []
    lines.append("# Final Compliance Report")
    lines.append("")
    lines.append(f"Generated: {_get(data, 'generated_at', datetime.now().isoformat())}")
    lines.append("")

    # Overall Summary
    e1s = _get(e1, "summary", {})
    e2s = _get(e2, "summary", {})
    e3s = _get(e3, "summary", {})
    overall_rows = [
        [
            "Exercise 1 - Space Compliance",
            _get(e1s, "checked_spaces", "-"),
            _get(e1s, "passed_count", "-"),
            _get(e1s, "failed_count", "-"),
            _get(e1s, "warning_count", "-"),
            _get(e1s, "compliance", "-"),
        ],
        [
            "Exercise 2 - Window Compliance",
            _get(e2s, "total_spaces_checked", "-"),
            _get(e2s, "passed_spaces", "-"),
            _get(e2s, "failed_spaces", "-"),
            _get(e2s, "warning_checks", "-"),
            _get(e2s, "overall_compliance", "-"),
        ],
        [
            "Bonus - Fire Safety",
            _get(e3s, "total_checks", "-"),
            _get(e3s, "passed_checks", "-"),
            _get(e3s, "failed_checks", "-"),
            _get(e3s, "warning_checks", "-"),
            _get(e3s, "overall_compliance", "-"),
        ],
    ]
    lines.append("## Overall Summary")
    lines.append(_md_table(
        ["Section", "Checked", "Passed", "Failed", "Warnings", "Overall"],
        overall_rows
    ))

    # Exercise 1
    lines.append("## Exercise 1 - Space Compliance")
    lines.append("### Passed Spaces")
    e1_passed = _get(e1, "passed", [])
    rows = [[
        s.get("space_id"), s.get("name"), s.get("space_type"),
        s.get("area"), s.get("height"),
        s.get("required_min_area"), s.get("required_min_height"),
        "Meets all checks"
    ] for s in e1_passed]
    lines.append(_md_table(
        ["ID", "Name", "Type", "Area m2", "Height m", "Req Area", "Req Height", "Reason"],
        rows
    ))

    lines.append("### Failed Spaces")
    e1_failed = _get(e1, "failed", [])
    rows = [[
        s.get("space_id"), s.get("name"), s.get("space_type"),
        s.get("area"), s.get("height"),
        s.get("required_min_area"), s.get("required_min_height"),
        _space_fail_reasons(s)
    ] for s in e1_failed]
    lines.append(_md_table(
        ["ID", "Name", "Type", "Area m2", "Height m", "Req Area", "Req Height", "Failure Reason"],
        rows
    ))

    lines.append("### Warnings")
    e1_warn = _get(e1, "warnings", [])
    rows = [[w.get("space_id"), w.get("name"), w.get("reason")] for w in e1_warn]
    lines.append(_md_table(["ID", "Name", "Warning"], rows))

    # Exercise 2
    lines.append("## Exercise 2 - Window Compliance")
    lines.append("### Passed Checks")
    e2_pass = _get(e2, "passed_checks_details", [])
    rows = [[c.get("check"), c.get("scope"), c.get("entity_id"), c.get("entity_name"), c.get("reason")] for c in e2_pass]
    lines.append(_md_table(["Check", "Scope", "Entity ID", "Entity", "Reason"], rows))

    lines.append("### Failed Checks")
    e2_fail = _get(e2, "failed_checks_details", [])
    rows = [[c.get("check"), c.get("scope"), c.get("entity_id"), c.get("entity_name"), c.get("reason")] for c in e2_fail]
    lines.append(_md_table(["Check", "Scope", "Entity ID", "Entity", "Reason"], rows))

    lines.append("### Passed Spaces")
    e2_ps = _get(e2, "passed_spaces_details", [])
    rows = [[
        s.get("space_id"), s.get("space_name"), s.get("window_count"),
        s.get("window_to_floor_ratio"), s.get("compliant"),
        "; ".join([c.get("reason", "") for c in s.get("checks", []) if c.get("status") == "PASS"])
    ] for s in e2_ps]
    lines.append(_md_table(
        ["ID", "Space", "Windows", "Ratio", "Compliant", "Pass Reasons"],
        rows
    ))

    lines.append("### Failed Spaces")
    e2_fs = _get(e2, "failed_spaces_details", [])
    rows = [[
        s.get("space_id"), s.get("space_name"), s.get("window_count"),
        s.get("window_to_floor_ratio"), s.get("compliant"),
        _space_fail_reasons(s)
    ] for s in e2_fs]
    lines.append(_md_table(
        ["ID", "Space", "Windows", "Ratio", "Compliant", "Failure Reasons"],
        rows
    ))

    # Bonus Fire Safety
    lines.append("## Bonus - Fire Safety")
    lines.append("### Longest Route")
    lr = _get(e3, "longest_route", {})
    rows = [[
        _get(lr, "from_space_name", "-"),
        _get(lr, "to_exit_space_name", "-"),
        " -> ".join(_get(lr, "path_space_names", [])) if _get(lr, "path_space_names", []) else "-",
        _get(lr, "distance_m", "-")
    ]]
    lines.append(_md_table(["From", "To Exit", "Path", "Distance m"], rows))

    lines.append("### Passed Checks")
    e3_pass = _get(e3, "passed_checks_details", [])
    rows = [[c.get("check"), c.get("scope"), c.get("entity_id"), c.get("entity_name"), c.get("reason")] for c in e3_pass]
    lines.append(_md_table(["Check", "Scope", "Entity ID", "Entity", "Reason"], rows))

    lines.append("### Failed Checks")
    e3_fail = _get(e3, "failed_checks_details", [])
    rows = [[c.get("check"), c.get("scope"), c.get("entity_id"), c.get("entity_name"), c.get("reason")] for c in e3_fail]
    lines.append(_md_table(["Check", "Scope", "Entity ID", "Entity", "Reason"], rows))

    lines.append("### Warnings")
    e3_warn = _get(e3, "warnings", [])
    rows = [[c.get("check"), c.get("scope"), c.get("entity_id"), c.get("entity_name"), c.get("reason")] for c in e3_warn]
    lines.append(_md_table(["Check", "Scope", "Entity ID", "Entity", "Warning"], rows))

    return "\n".join(lines)


# Load existing combined JSON report (from final_report.txt) if not already in memory
if "combined_report" in globals():
    report_data = combined_report
else:
    with open("final_report.txt", "r", encoding="utf-8") as f:
        report_data = json.load(f)

readable = build_readable_report(report_data)

out_md = "final_report_readable.md"
with open(out_md, "w", encoding="utf-8") as f:
    f.write(readable)

print("Readable report saved to:", os.path.abspath(out_md))
print("Tip: open Markdown preview in VS Code for table view.")


Readable report saved to: c:\Users\Asus\github-classroom\iaac-maai\intro-to-ifc-susumit08-hue\final_report_readable.md
Tip: open Markdown preview in VS Code for table view.


## Useful IFC Concepts

### Common Entity Types

- **IfcSpace**: A room, area, or zone in the building
- **IfcWindow**: Windows for light and ventilation  
- **IfcDoor**: Doors, openings, access points
- **IfcWall**: Boundary elements
- **IfcBuildingElement**: General building components
- **IfcElement**: Physical building pieces with properties

### Accessing Properties

```python
# Get all entities of a type
elements = ifc.by_type("IfcWindow")

# Access properties
entity.Name                    # String name
entity.Description             # Text description
entity.ObjectPlacement        # Location/coordinates
entity.Representation         # Geometry data
entity.QuantityInSpace         # Calculate-derived quantities

# Common relationships
entity.HasProperties           # Get properties
entity.BoundedBy               # Spatial boundaries
entity.HostedBy                # Connection relationships
```

### Helpful Methods

```python
# Query by GUID
entity = ifc.by_id(guid)

# Filter by property value
elements = ifc.by_attribute("Name", "Kitchen")

# Get all instances of a type
spaces = ifc.by_type("IfcSpace")

# Check entity type
if space.is_a("IfcSpace"):
    print("This is a space")
```

## Resources for Help

- **IFC Knowledge Base**: https://notebooklm.google.com/notebook/0925c2a1-519b-40a8-aca4-1e832d219f3c
- **IfcOpenShell Documentation**: https://docs.ifcopenshell.org/ifcopenshell-python.html
- **BuildingSmart Standards**: https://www.buildingsmart.org/
- **Catalan Building Code**: https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e

## Next Steps

After completing these exercises, you'll have learned:
- ✓ How to load and explore IFC files
- ✓ How to extract spatial and building data
- ✓ How to validate designs against building codes
- ✓ How to work with doors, windows, and routes

These skills apply to real-world AEC (Architecture, Engineering, Construction) workflows.